# 07 — Business Insights & Recommendations

Synthesizing findings from SQL analysis, EDA, modeling, and SHAP into 
actionable insights - the "so what" of the whole project.

In [1]:
import pandas as pd
df = pd.read_csv('../data/clean_labeled_repos.csv')

print("Total repos analyzed:", len(df))
print("High risk:", (df['risk_category']=='High').sum(), f"({(df['risk_category']=='High').mean():.1%})")
print("Medium risk:", (df['risk_category']=='Medium').sum(), f"({(df['risk_category']=='Medium').mean():.1%})")
print("Low risk:", (df['risk_category']=='Low').sum(), f"({(df['risk_category']=='Low').mean():.1%})")
print()
print("Archived:", df['is_archived'].sum())
print("No license:", (df['license']=='No license').sum(), f"({(df['license']=='No license').mean():.1%})")

Total repos analyzed: 498
High risk: 162 (32.5%)
Medium risk: 86 (17.3%)
Low risk: 250 (50.2%)

Archived: 31
No license: 80 (16.1%)


## Key Findings

Of the 498 repos analyzed, **32.5% (162) are High risk**, 17.3% (86) are 
Medium risk, and 50.2% (250) are Low risk. Only 31 are formally archived — 
meaning the vast majority of at-risk repos are *quietly* dormant rather 
than explicitly flagged, which is exactly the gap this project targets: 
finding risk before a maintainer bothers to archive it (or before nobody's 
left to do so).

---

### 1. Popularity is not a proxy for health

**Question:** Does star count protect a project from risk?

**Finding:** Risk rate rises sharply and monotonically as popularity drops - 
3% (mega, >50k stars) → 8% (large) → 31% (mid) → 53% (small) → 68% (tiny). 
But even 3 of 100 mega-popular repos are High risk, including 
`AUTOMATIC1111/stable-diffusion-webui` (164k+ stars, 2+ years dormant) and 
`CyC2018/CS-Notes` (185k+ stars, 3+ years dormant).

**Business implication:** Star count is a reasonable *first-pass filter* 
for triage, but not a substitute for actual health monitoring - a 
dependency-scanning tool that only checks "is this popular" would still 
miss real risk in prominent projects.

---

### 2. Missing a license is a stronger risk signal than most people would guess

**Finding:** Repos with no license file average a 61.4 risk score vs. 30.8 
for licensed repos - roughly double - and are 2x more likely to be 
High risk (60% vs. 27.3%). Confirmed independently through SHAP 
attribution on the trained model, not just aggregate statistics.

**Business implication:** For an organization evaluating whether to depend 
on an open-source library, "does it have a license" is a fast, free, 
zero-API-calls-required signal worth checking before anything else - and 
it correlates with deeper health issues, not just a legal technicality.

---

### 3. Bus factor matters independent of recent activity

**Finding:** Average risk score drops steadily from ~79 (0-1 recent 
contributors) to ~15 (16+ contributors). This holds even after fixing a 
labeling bug where GitHub's API undercounts contributors when commit 
emails aren't linked to a GitHub account (affected 7/498 repos, including 
a 316k-star actively-maintained repo that was nearly mislabeled).

**Business implication:** A project maintained by one person is fragile 
regardless of how active that person currently is - succession risk is a 
real, measurable dimension separate from "is there activity right now."

---

### 4. Raw open-issue count is actively misleading as a risk signal

**Finding:** Counterintuitively, Low-risk repos have the *most* open 
issues (median 89.5) and High-risk repos the fewest (median 6) - because 
dead projects stop attracting new issue filers entirely. A naive "lots of 
open issues = neglected project" heuristic gets this backwards.

**Business implication:** Simple issue-count dashboards (a common feature 
in dependency-management tools) can mislead teams into deprioritizing 
genuinely at-risk projects that just look "quiet" rather than "overloaded."

---

### 5. Cheap metadata gets you most of the way there - but not all the way

**Finding:** A model using only stars, forks, language, license, and age 
(no commit-history crawling required) achieves 71-72% accuracy predicting 
risk category. Adding deeper signals (PR/issue history, contributor 
counts) doesn't move overall accuracy, but meaningfully improves detection 
of Medium-risk repos specifically (F1 +0.10, recall 0.29 → 0.47) - the 
ambiguous, hardest-to-classify middle ground.

**Business implication:** A lightweight, low-cost screening pass 
(cheap features only) can flag obvious cases efficiently at scale; deeper 
API-based analysis is worth reserving for the harder, more ambiguous 
cases rather than crawling every repo's full history by default.

---

### 6. Predictions are individually explainable, not just statistically accurate

**Finding:** SHAP analysis shows individual, named examples with full 
reasoning - e.g. `myqianlan/antd-admin-boilerplate` flagged at 98.5% 
High-risk confidence (100 stars, 6+ years dormant, zero merged PRs, no 
license), contrasted with `bevyengine/bevy` at 97% Low-risk confidence 
(48k stars, 8,722 merged PRs, 1,546 contributors).

**Business implication:** A real triage tool built on this approach 
could tell a team not just "this dependency is risky" but specifically 
*why* - enabling a more informed decision than a single opaque score.

## Recommendations

If this analysis were used by an engineering team deciding which 
open-source dependencies to adopt or continue relying on, the practical 
takeaways would be:

1. **Don't rely on stars alone.** Use it as a first filter, but always 
   check at least one activity signal (last commit date, recent PR/issue 
   volume) before treating a popular repo as automatically safe.

2. **Treat "no license" as a red flag worth investigating**, not just a 
   legal checkbox - it's one of the cheapest, fastest checks available 
   and correlates with deeper maintenance issues.

3. **Ask "how many people would need to disappear to kill this project?"** 
   - not just "is someone active right now." A single-maintainer project 
   is a standing risk even mid-crisis, before anything visibly breaks.

4. **Don't use open issue count as a health metric on its own.** A quiet 
   issue tracker can mean either "well-maintained, few problems" or 
   "abandoned, nobody's even filing bugs anymore" - it needs to be read 
   alongside recent activity, not in isolation.

5. **Use a two-tier screening approach in practice.** Run cheap metadata 
   checks (stars, license, language, age) across an entire dependency 
   tree cheaply and at scale; reserve deeper API-based analysis (commit 
   history, contributor patterns) for the ambiguous middle cases the 
   cheap pass can't confidently resolve - since that's specifically 
   where the extra signal earned its keep in this analysis.

6. **Prefer tools that explain *why* a dependency is flagged**, not just 
   a risk score - a number alone doesn't tell a team whether the concern 
   is "single maintainer," "no recent commits," or "no license," and 
   each of those calls for a different response.

## Limitations & Future Improvements

- **Single point-in-time snapshot.** This dataset captures repo state at 
  one moment; a longitudinal version tracking the same repos over months 
  would allow genuine forecasting ("will this repo go dormant in the next 
  6 months") rather than describing current state.
- **Risk label is rule-based, not ground truth.** `is_archived` is the 
  only true external label available; the composite `risk_score` is a 
  reasonable, validated proxy but ultimately a judgment call, not an 
  objective fact.
- **Sample size (498 repos) limits deeper modeling.** A larger sample, 
  especially more Medium-risk examples specifically, would likely improve 
  the model's weakest area.
- **Language/risk correlation is likely confounded** by which kinds of 
  projects tend to get written in which languages, not a causal 
  relationship - worth deeper investigation with a controlled comparison 
  (e.g. matching similar-age, similar-purpose repos across languages).